In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
class KaplanScalingLaw:
    """Kaplan et al. (2020) Scaling Law"""
    
    def __init__(self):
        # 경험적 파라미터 (OpenAI GPT-3 기반)
        self.alpha = 0.07  # 모델 스케일링 지수
        self.beta = 0.16   # 데이터 스케일링 지수
        self.a = 0.5
        self.b = 0.5
        self.c = 0.5
    
    def compute_loss(self, N, D):
        """손실 계산: L(N,D) = a*N^(-alpha) + b*D^(-beta) + c"""
        return self.a * (N ** (-self.alpha)) + self.b * (D ** (-self.beta)) + self.c
    
    def optimal_allocation(self, compute_budget):
        """compute_budget에서 최적 N, D 계산
        
        계산량: C ≈ 6 * N * D
        최적: N ≈ D (Kaplan의 권장)
        """
        # C ≈ 6 * N * D에서 N=D로 최적화
        # C ≈ 6 * N^2 → N ≈ sqrt(C/6)
        optimal_N = np.sqrt(compute_budget / 6.0)
        optimal_D = np.sqrt(compute_budget / 6.0)
        
        return optimal_N, optimal_D

kaplan = KaplanScalingLaw()
print(f"Kaplan Scaling Law 초기화")
print(f"  alpha (모델): {kaplan.alpha}")
print(f"  beta (데이터): {kaplan.beta}")

Kaplan Scaling Law 초기화
  alpha (모델): 0.07
  beta (데이터): 0.16


In [4]:
class ChinchillaScalingLaw:
    """DeepMind Chinchilla (2022) Scaling Law"""
    
    def __init__(self):
        # 경험적 파라미터 (더 정확한 추정)
        self.alpha = 0.07  # 모델 스케일링 지수
        self.beta = 0.16   # 데이터 스케일링 지수
        self.A = 0.5
        self.B = 0.5
        self.E = 0.5
    
    def compute_loss(self, N, D):
        """손실 계산: L(N,D) = E + A/N^alpha + B/D^beta"""
        return self.E + self.A / (N ** self.alpha) + self.B / (D ** self.beta)
    
    def optimal_allocation(self, compute_budget):
        """compute_budget에서 최적 N, D 계산
        
        최적 비율: D ≈ 20 * N
        계산량: C ≈ 6 * N * D ≈ 6 * N * 20N = 120 * N^2
        → N ≈ sqrt(C/120)
        """
        optimal_N = np.sqrt(compute_budget / 120.0)
        optimal_D = 20.0 * optimal_N
        
        return optimal_N, optimal_D

chinchilla = ChinchillaScalingLaw()
print(f"Chinchilla Scaling Law 초기화")
print(f"  alpha (모델): {chinchilla.alpha}")
print(f"  beta (데이터): {chinchilla.beta}")
print(f"  최적 비율: D = 20 * N")

Chinchilla Scaling Law 초기화
  alpha (모델): 0.07
  beta (데이터): 0.16
  최적 비율: D = 20 * N


In [5]:
# 다양한 컴퓨팅 예산 시나리오
budget_scenarios = [
    ("소규모", 1e15),      # 1000 TFLOP-s (작은 조직)
    ("중규모", 1e17),      # 100,000 TFLOP-s (중간 회사)
    ("대규모", 1e19),      # 10,000,000 TFLOP-s (대기업)
    ("초대규모", 1e21),    # 1,000,000,000 TFLOP-s (GPT-3 수준)
]

results = []

for scenario_name, compute_budget in budget_scenarios:
    # Kaplan 최적
    n_kaplan, d_kaplan = kaplan.optimal_allocation(compute_budget)
    loss_kaplan = kaplan.compute_loss(n_kaplan, d_kaplan)
    
    # Chinchilla 최적
    n_chinchilla, d_chinchilla = chinchilla.optimal_allocation(compute_budget)
    loss_chinchilla = chinchilla.compute_loss(n_chinchilla, d_chinchilla)
    
    results.append({
        '시나리오': scenario_name,
        'Kaplan_모델': f"{n_kaplan:.2e}",
        'Kaplan_데이터': f"{d_kaplan:.2e}",
        'Kaplan_손실': f"{loss_kaplan:.4f}",
        'Chinchilla_모델': f"{n_chinchilla:.2e}",
        'Chinchilla_데이터': f"{d_chinchilla:.2e}",
        'Chinchilla_손실': f"{loss_chinchilla:.4f}",
        '성능차': f"{((loss_kaplan - loss_chinchilla) / loss_kaplan * 100):.1f}%"
    })

df_results = pd.DataFrame(results)
print("\n" + "="*100)
print("[Kaplan vs Chinchilla Scaling Law 비교]")
print("="*100)
print(df_results.to_string(index=False))


[Kaplan vs Chinchilla Scaling Law 비교]
시나리오 Kaplan_모델 Kaplan_데이터 Kaplan_손실 Chinchilla_모델 Chinchilla_데이터 Chinchilla_손실   성능차
 소규모  1.29e+07   1.29e+07    0.6953      2.89e+06       5.77e+07        0.7051 -1.4%
 중규모  1.29e+08   1.29e+08    0.6605      2.89e+07       5.77e+08        0.6700 -1.5%
 대규모  1.29e+09   1.29e+09    0.6326      2.89e+08       5.77e+09        0.6416 -1.4%
초대규모  1.29e+10   1.29e+10    0.6101      2.89e+09       5.77e+10        0.6183 -1.4%


In [ ]:
# 실제 LLM 모델들
actual_models = pd.DataFrame([
    {'이름': 'GPT-2', '파라미터': 1.5e9, '훈련토큰': 4.0e10, '연도': 2019},
    {'이름': 'GPT-3', '파라미터': 1.75e11, '훈련토큰': 3.0e11, '연도': 2020},
    {'이름': 'Chinchilla', '파라미터': 7.0e10, '훈련토큰': 1.4e12, '연도': 2022},
    {'이름': 'LLaMA-7B', '파라미터': 7.0e9, '훈련토큰': 1.0e12, '연도': 2023},
    {'이름': 'LLaMA-65B', '파라미터': 6.5e10, '훈련토큰': 1.4e12, '연도': 2023},
])

# 비율 계산
actual_models['D/N 비율'] = actual_models['훈련토큰'] / actual_models['파라미터']
actual_models['계산량'] = 6 * actual_models['파라미터'] * actual_models['훈련토큰']

print("\n" + "="*100)
print("[실제 LLM 모델 분석]")
print("="*100)
for idx, row in actual_models.iterrows():
    print(f"\n{row['이름']} ({row['연도']})")
    print(f"  파라미터: {row['파라미터']:.2e}")
    print(f"  훈련토큰: {row['훈련토큰']:.2e}")
    print(f"  D/N 비율: {row['D/N 비율']:.1f}x")
    print(f"  평가: ", end="")
    if row['D/N 비율'] < 5:
        print(" 데이터 부족 (과적합 위험)")
    elif 5 <= row['D/N 비율'] <= 25:
        print(" 적절한 비율")
    else:
        print("데이터 충분 (더 큰 모델 고려)")